In [105]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split

In [106]:
#%cd ../..
#!ls

In [107]:
tmp_data = pd.read_csv('EDA/data/findata-2.csv', index_col=0)
#tmp_data = pd.read_excel('EDA/data/dataset.xlsx', index_col=0)

In [108]:
tmp_data.info(show_counts=True, verbose=True)

<class 'pandas.core.frame.DataFrame'>
Index: 1001 entries, 0 to 1000
Data columns (total 79 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   IC50, mM             1001 non-null   float64
 1   CC50, mM             1001 non-null   float64
 2   SI                   1001 non-null   float64
 3   MaxAbsEStateIndex    1001 non-null   float64
 4   MaxEStateIndex       1001 non-null   float64
 5   MinAbsEStateIndex    1001 non-null   float64
 6   MinEStateIndex       1001 non-null   float64
 7   qed                  1001 non-null   float64
 8   SPS                  1001 non-null   float64
 9   MolWt                1001 non-null   float64
 10  HeavyAtomMolWt       1001 non-null   float64
 11  ExactMolWt           1001 non-null   float64
 12  NumValenceElectrons  1001 non-null   int64  
 13  MaxPartialCharge     1001 non-null   float64
 14  MinPartialCharge     1001 non-null   float64
 15  MaxAbsPartialCharge  1001 non-null   float6

In [109]:
go_data = tmp_data.copy()

In [110]:
# Удалим не нужные тут таргеты 
go_data = go_data.drop(columns=['CC50, mM', 'IC50, mM'])

go_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1001 entries, 0 to 1000
Data columns (total 77 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   SI                   1001 non-null   float64
 1   MaxAbsEStateIndex    1001 non-null   float64
 2   MaxEStateIndex       1001 non-null   float64
 3   MinAbsEStateIndex    1001 non-null   float64
 4   MinEStateIndex       1001 non-null   float64
 5   qed                  1001 non-null   float64
 6   SPS                  1001 non-null   float64
 7   MolWt                1001 non-null   float64
 8   HeavyAtomMolWt       1001 non-null   float64
 9   ExactMolWt           1001 non-null   float64
 10  NumValenceElectrons  1001 non-null   int64  
 11  MaxPartialCharge     1001 non-null   float64
 12  MinPartialCharge     1001 non-null   float64
 13  MaxAbsPartialCharge  1001 non-null   float64
 14  MinAbsPartialCharge  1001 non-null   float64
 15  FpDensityMorgan1     1001 non-null   float6

In [111]:
# Выбираем самые полезные параметры 

# Рассчитываем корреляцию всех признаков 
go_correlations = go_data.corr()['SI'].abs().sort_values()

# Отбираем признаки с корреляцией больше 0.1 
gl_high_info_features = go_correlations[go_correlations > 0.05]

print("Информативные признаки (есть связь):")
gl_high_info_features = gl_high_info_features.drop(['SI'], errors='ignore')


#Для финальной модели оставляем только информативные параметры  
gl_final_param = gl_high_info_features.index.unique().tolist() 

print(len(gl_final_param))
display(go_correlations.sort_values(ascending=False).head(100))

display(gl_final_param)

Информативные признаки (есть связь):
24


SI                   1.000000
BalabanJ             0.162955
RingCount            0.124955
NumAliphaticRings    0.093939
SlogP_VSA6           0.089492
                       ...   
MaxEStateIndex       0.005866
SPS                  0.004658
Ipc                  0.003923
Kappa1               0.003600
VSA_EState7          0.000095
Name: SI, Length: 77, dtype: float64

['SMR_VSA10',
 'FpDensityMorgan2',
 'MinAbsPartialCharge',
 'Chi4n',
 'Kappa3',
 'MinAbsEStateIndex',
 'MinPartialCharge',
 'Chi3v',
 'BCUT2D_LOGPLOW',
 'SMR_VSA5',
 'Chi4v',
 'PEOE_VSA6',
 'FractionCSP3',
 'BertzCT',
 'EState_VSA8',
 'BCUT2D_LOGPHI',
 'MolLogP',
 'AvgIpc',
 'VSA_EState4',
 'FpDensityMorgan1',
 'SlogP_VSA6',
 'NumAliphaticRings',
 'RingCount',
 'BalabanJ']

In [112]:
#перебором выявим параметры котрые коррелируют между собой > 60% и оставим только второй 

for col_1 in gl_final_param:
    for col_2 in gl_final_param:
        if col_1 != col_2:
            lv_correlation = go_data[col_1].corr(go_data[col_2])
            if lv_correlation >= 0.70:
                print(f' Параметр {col_1} коррелирует с парамтером {col_2} : {lv_correlation}')
                gl_final_param.remove(col_2)
display(gl_final_param)

 Параметр FpDensityMorgan2 коррелирует с парамтером FpDensityMorgan1 : 0.947756641032178
 Параметр Chi4n коррелирует с парамтером Chi3v : 0.9369290597451744
 Параметр Chi4n коррелирует с парамтером Chi4v : 0.9663789355818365
 Параметр Chi4n коррелирует с парамтером NumAliphaticRings : 0.7104343085012806
 Параметр BertzCT коррелирует с парамтером AvgIpc : 0.779775515975152
 Параметр BertzCT коррелирует с парамтером RingCount : 0.7549455688994523


['SMR_VSA10',
 'FpDensityMorgan2',
 'MinAbsPartialCharge',
 'Chi4n',
 'Kappa3',
 'MinAbsEStateIndex',
 'MinPartialCharge',
 'BCUT2D_LOGPLOW',
 'SMR_VSA5',
 'PEOE_VSA6',
 'FractionCSP3',
 'BertzCT',
 'EState_VSA8',
 'BCUT2D_LOGPHI',
 'MolLogP',
 'VSA_EState4',
 'SlogP_VSA6',
 'BalabanJ']

In [120]:
go_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1001 entries, 0 to 1000
Data columns (total 77 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   SI                   1001 non-null   float64
 1   MaxAbsEStateIndex    1001 non-null   float64
 2   MaxEStateIndex       1001 non-null   float64
 3   MinAbsEStateIndex    1001 non-null   float64
 4   MinEStateIndex       1001 non-null   float64
 5   qed                  1001 non-null   float64
 6   SPS                  1001 non-null   float64
 7   MolWt                1001 non-null   float64
 8   HeavyAtomMolWt       1001 non-null   float64
 9   ExactMolWt           1001 non-null   float64
 10  NumValenceElectrons  1001 non-null   int64  
 11  MaxPartialCharge     1001 non-null   float64
 12  MinPartialCharge     1001 non-null   float64
 13  MaxAbsPartialCharge  1001 non-null   float64
 14  MinAbsPartialCharge  1001 non-null   float64
 15  FpDensityMorgan1     1001 non-null   float6

In [121]:
X = go_data[gl_final_param]
#X = go_data.drop(columns='IC50, mM')
y = pd.DataFrame(np.log1p(go_data['SI']),columns=['SI'])

In [122]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

print(f'Train dataset size: {X_train.shape}, {y_train.shape}')
print(f'Test dataset size: {X_test.shape}, {y_test.shape}')

Train dataset size: (700, 18), (700, 1)
Test dataset size: (301, 18), (301, 1)


In [123]:
import warnings
warnings.filterwarnings('ignore')

In [124]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

models = {
    "LinearRegression": (LinearRegression(), 
        {
            'fit_intercept': [True, False]
        }),

    "RidgeRegression": (Ridge(), 
        {
            'alpha': [0.1, 1.0, 10.0, 100.0],  
            'solver': ['auto', 'cholesky', 'sag', 'lsqr']
        }),

    "RandomForestRegressor": (RandomForestRegressor(), 
        {
            'n_estimators': (10, 200, 350, 500, 1000),
            'max_depth': (3, 10, 25,100, 500),
            'min_samples_split': (2, 10, 15, 50, 100, 500)

        }),


    "GradientBoostingRegressor": (GradientBoostingRegressor(), 
        {
            'n_estimators': (10, 200, 350, 500, 1000),
            'learning_rate': [0.05, 0.1, 0.5],
            'max_depth': [3, 5, 9, 15, 50]

        })
    
}

In [118]:
from skopt import BayesSearchCV
from sklearn.metrics import silhouette_score
from sklearn.model_selection import PredefinedSplit


# Перебор моделей
best_global_score = -10
best_model = None
results_report = []


for name, (model, params) in models.items():
    print(f"Обучаем {name}...")

    # Байесовская оптимизация гиперпараметров
    bayes_search = BayesSearchCV(
        estimator=model,
        search_spaces=params,
        n_iter=35,
        cv=10,
        scoring='neg_mean_squared_error',  #neg_mean_absolute_error
        n_jobs=-1,
        random_state=42
    )

    # Обучение модели
    bayes_search.fit(X_train, y_train)

    # 
    score = bayes_search.best_score_  # type: ignore
    results_report.append({"Model": name, "Score": score, "Params": bayes_search.best_params_}) # type: ignore
    
    # Сохраняем абсолютного победителя
    if score > best_global_score:
        best_global_score = score
        best_model = bayes_search.best_estimator_ # type: ignore

# --- АНАЛИЗ ---
print("\n--- Report  ---")
print(pd.DataFrame(results_report))
print(f"\n Лучшая модель: {best_model}")



Обучаем LinearRegression...


Обучаем RidgeRegression...
Обучаем RandomForestRegressor...
Обучаем GradientBoostingRegressor...

--- Report  ---
                       Model     Score  \
0           LinearRegression -1.904833   
1            RidgeRegression -1.900556   
2      RandomForestRegressor -1.521141   
3  GradientBoostingRegressor -1.576118   

                                              Params  
0                            {'fit_intercept': True}  
1                   {'alpha': 1.0, 'solver': 'auto'}  
2  {'max_depth': 100, 'min_samples_split': 10, 'n...  
3  {'learning_rate': 0.05, 'max_depth': 3, 'n_est...  

 Лучшая модель: RandomForestRegressor(max_depth=100, min_samples_split=10, n_estimators=1000)


In [119]:
# на тестовой выборке 
from sklearn import metrics

y_pred = best_model.predict(X_test)  # type: ignore

print("MAE", metrics.mean_absolute_error(y_test, y_pred))
print("MSE", metrics.mean_squared_error(y_test, y_pred))
print("R2 Score:", best_model.score(X_test, y_test)) # type: ignore

MAE 0.963427436927165
MSE 1.5497033384975336
R2 Score: 0.29979864583167715
